# Data Cleaning

### The dataset contained missing values, inconsistent formatting, and missing entries.

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r"C:\Users\Mika-iel Osman\OneDrive\Documents\Cleaning Data\dirty_cafe_sales.csv")

In [4]:
pd.set_option('display.max_rows', None)

In [5]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [7]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [8]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [9]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [14]:
df['item'] = df['item'].str.strip().str.lower().fillna('unknown').str.replace('error' , 'unknown' )
df['item'].unique()

array(['coffee', 'cake', 'cookie', 'salad', 'smoothie', 'unknown',
       'sandwich', 'juice', 'tea'], dtype=object)

In [139]:
df['quantity'] = pd.to_numeric(df['quantity'].astype(str).str.strip().str.replace(r'[^\d.]','', regex=True),errors = 'coerce')
df['quantity'] = df['quantity'].astype('Int64')

In [17]:
df['price_per_unit'] = pd.to_numeric(df['price_per_unit'].astype(str).str.strip().str.replace(r'[^\d.]','', regex=True),errors = 'coerce')

array([2. , 3. , 1. , 5. , 4. , 1.5, nan])

In [18]:
df['total_spent'] = pd.to_numeric(df['total_spent'].astype(str).str.strip().str.replace(r'[^\d.]','', regex=True),errors = 'coerce')

array([ 4. , 12. ,  nan, 10. , 20. ,  9. , 16. , 15. , 25. ,  8. ,  5. ,
        3. ,  6. ,  2. ,  1. ,  7.5,  4.5,  1.5])

In [19]:
df['payment_method'] = df['payment_method'].str.lower().replace('error','unknown')

array(['credit card', 'cash', 'unknown', 'digital wallet', nan],
      dtype=object)

In [20]:
df['location'] = df['location'].str.lower().str.replace('error','unknown')

array(['takeaway', 'in-store', 'unknown', nan], dtype=object)

In [22]:
df['transaction_date'] = df['transaction_date'].str.lower().str.replace('error','unknown').fillna('unknown')
df['transaction_date'] = df['transaction_date'].replace('unknown', np.nan)
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors = 'coerce')

In [147]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9946 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    9946 non-null   object        
 1   item              9946 non-null   object        
 2   quantity          9926 non-null   Int64         
 3   price_per_unit    9946 non-null   float64       
 4   total_spent       9447 non-null   float64       
 5   payment_method    7384 non-null   object        
 6   location          6704 non-null   object        
 7   transaction_date  9489 non-null   datetime64[ns]
dtypes: Int64(1), datetime64[ns](1), float64(2), object(4)
memory usage: 967.1+ KB


In [145]:
# check if the 'price_per_unit is correct throughout the sheet, if not change it
# coffee = 2 ; cake = 3; cookie = 1; salad = 5; smoothie = 4; sandwich = 4 ; juice = 3 ; tea = 1.5 
# fill in uknowns as well for items

price_map = {'coffee': 2, 'cake': 3, 'cookie': 1, 'salad': 5, 'smoothie': 4, 'sandwich': 4, 'juice': 3, 'tea': 1.5}
df['correct_price'] = df['item'].map(price_map)
df['is_correct'] = df['price_per_unit'] == df['correct_price']

# double check and fillna in 'price_per_unit' with the 'correct_price'
# df[(df['is_correct'] == False) & (df['item'] != 'unknown')][['item', 'price_per_unit', 'correct_price', 'is_correct']]
df['price_per_unit'] = df['price_per_unit'].fillna(df['correct_price'])
 

#to_drop = df[(df['item'] == 'unknown') & (df['price_per_unit'].isna())].index
#df = df.drop(to_drop)


#or drop columns here
#df = df.drop(df.columns[8:], axis = 1)

In [93]:
# fill in unknown 'item' where possible

price_map2 = { 2 :'coffee', 1 :'cookie', 5 : 'salad', 1.5 :'tea'}
df['correct_item'] = df['price_per_unit'].map(price_map2)
indexes_to_correct = df[(df['item'] == 'unknown') & (df['correct_item'].notna())][['item','correct_item']].index
df.loc[indexes_to_correct, 'item'] = df.loc[indexes_to_correct, 'correct_item']

,item,correct_item
6,unknown,NaN
8,unknown,NaN
36,unknown,NaN
61,unknown,NaN
69,unknown,NaN
91,unknown,NaN
153,unknown,NaN
165,unknown,NaN
168,unknown,NaN
184,unknown,NaN


In [106]:
# fill in Na and check if the 'total_spent' is correct

df['calculated_tot'] = df['quantity'] * df['price_per_unit']
df['total_spent'] = df[['total_spent']].fillna(df['calculated_tot'])

In [108]:
# drop columns and clear space
df = df.drop(columns = ['correct_price','is_correct','correct_item', 'calculated_tot'])

In [137]:
# fill in missing 'quantity'


df['quantity_calc'] = df['total_spent'] / df['price_per_unit']
#df[(df['quantity'] != df['quantity_calc'])]
#df[df['quantity'].isna()][['quantity','quantity_calc']] 
df['quantity'] = df['quantity'].astype(float)
df['quantity'] = df['quantity'].fillna(df['quantity_calc'])

In [162]:
#check and fill in the 'total_spent'

#df['total_spent_calc'] = df['quantity'] * df['price_per_unit']
#df['total_spent'] = df['total_spent'].fillna(df['total_spent_calc'])
#df[(df['total_spent'] != df['total_spent_calc'])]

#drop column
#df = df.drop('total_spent_calc', axis = 1)

In [165]:
df['location'] = df['location'].fillna('unknown')

In [179]:
# convert Unknowns to na as it is easier to work with
#check

df.isna().sum()
#df = df.replace('unknown', np.nan)

transaction_id        0
item                447
quantity              0
price_per_unit        0
total_spent           0
transaction_date    457
dtype: int64

In [174]:
#dropping na rows in 'quantity' and 'total_spent'
#df = df.dropna(subset=['quantity', 'total_spent'])

In [178]:
df = df.drop(columns = ['payment_method','location'])

In [182]:
df.to_csv(r'C:\Users\Mika-iel Osman\OneDrive\Documents\Cleaning Data\Cleaning_data_completed\Cleaned_cafe_data.csv', index = False)